# Import + Dataset

In [1]:
import kagglehub
import numpy as np
import pandas as pd
import pyspark as ps
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

customers = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_customers_dataset.csv')
geolocation = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_geolocation_dataset.csv')
order_items = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_order_items_dataset.csv')
order_payments = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_order_payments_dataset.csv')
order_review = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_order_reviews_dataset.csv')
orders = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_orders_dataset.csv')
products = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_products_dataset.csv')
sellers = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\olist_sellers_dataset.csv')
category_name = pd.read_csv(r'C:\\Users\\Gustavo\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2\\product_category_name_translation.csv')

# Ajuste base de dados

## Clientes

In [3]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [4]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
customers_na = customers.isna().sum()
customers_na

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

## Itens do Pedido

In [6]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [7]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [9]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

## Pagamento(s) do Pedido

In [10]:
order_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [11]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [12]:
order_payments_na = order_payments.isna().sum()
order_payments_na

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

## Review do Pedido

In [13]:
order_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [14]:
order_review.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [15]:
order_review_na = order_review.isna().sum()
order_review_na

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [16]:
order_review['review_creation_date'] = pd.to_datetime(order_review['review_creation_date'])
order_review['review_answer_timestamp'] = pd.to_datetime(order_review['review_answer_timestamp'])

In [17]:
order_review['review_comment_title'] = order_review['review_comment_title'].fillna('Avaliação sem titulo.')
order_review['review_comment_message'] = order_review['review_comment_message'].fillna('Avaliação sem texto.')

In [18]:
order_review.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,Avaliação sem titulo.,Avaliação sem texto.,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,Avaliação sem titulo.,Avaliação sem texto.,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,Avaliação sem titulo.,Avaliação sem texto.,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Avaliação sem titulo.,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Avaliação sem titulo.,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [19]:
duplicados = order_review[order_review.duplicated(subset=['review_id'], keep=False)]
duplicados.sort_values('review_id')

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,Avaliação sem titulo.,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,Avaliação sem titulo.,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,Avaliação sem titulo.,Avaliação sem texto.,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,Avaliação sem titulo.,Avaliação sem texto.,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,Avaliação sem titulo.,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,Avaliação sem titulo.,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,Avaliação sem titulo.,Avaliação sem texto.,2018-03-17,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,Avaliação sem titulo.,Avaliação sem texto.,2018-03-17,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,Avaliação sem titulo.,Avaliação sem texto.,2017-07-22,2017-07-26 13:41:07


In [20]:
mean_review_score = duplicados['review_score'].mean()
print(f'{(mean_review_score):.2f}')

3.80


Há um problema de duplicação do ID da review, mas que está levando a pedidos de compras diferente. Isso pode ocorrer por alguns motivos, mas dentre os principais estão:

1. Erros/Bugs de sistema (review duplicada em outro pedido);
2. Aplicação proposital.

Acreditamos que seja a primeira opção, visto que apesar do nível de linhas duplicadas (1603), a média de avaliações não é alta como 4 ou 5 por exemplo, mas sim 3,80. Portanto, vamos utilizar do método drop_duplicates() para excluir todas as ocorrências após a primeira aparição no banco de dados.

In [21]:
order_review = order_review.drop_duplicates(subset=['review_id'], keep='first')

## Pedidos

In [22]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [23]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [24]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [25]:
orders_na = orders.isna().sum()
orders_na

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## Produtos

In [26]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [27]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [28]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [29]:
columns_adjust=[
    'product_name_lenght', 
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

products[columns_adjust] = products[columns_adjust].fillna(0)                           

products['product_name_lenght'] = products['product_name_lenght'].astype(int) 
products['product_description_lenght'] = products['product_description_lenght'].astype(int) 
products['product_photos_qty'] = products['product_photos_qty'].astype(int)

products.info()
#611 produtos com 0 'em tese'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32951 non-null  int32  
 3   product_description_lenght  32951 non-null  int32  
 4   product_photos_qty          32951 non-null  int32  
 5   product_weight_g            32951 non-null  float64
 6   product_length_cm           32951 non-null  float64
 7   product_height_cm           32951 non-null  float64
 8   product_width_cm            32951 non-null  float64
dtypes: float64(4), int32(3), object(2)
memory usage: 1.9+ MB


In [30]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght             0
product_description_lenght      0
product_photos_qty              0
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64

In [31]:
products['product_category_name'] = products['product_category_name'].fillna('desconhecido')

In [32]:
products.isna().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

## Vendedores

In [33]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [34]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [35]:
sellers_na = sellers.isna().sum()
sellers_na

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

## Categoria

In [36]:
category_name.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [37]:
category_name['product_category_name'].unique()

array(['beleza_saude', 'informatica_acessorios', 'automotivo',
       'cama_mesa_banho', 'moveis_decoracao', 'esporte_lazer',
       'perfumaria', 'utilidades_domesticas', 'telefonia',
       'relogios_presentes', 'alimentos_bebidas', 'bebes', 'papelaria',
       'tablets_impressao_imagem', 'brinquedos', 'telefonia_fixa',
       'ferramentas_jardim', 'fashion_bolsas_e_acessorios',
       'eletroportateis', 'consoles_games', 'audio', 'fashion_calcados',
       'cool_stuff', 'malas_acessorios', 'climatizacao',
       'construcao_ferramentas_construcao',
       'moveis_cozinha_area_de_servico_jantar_e_jardim',
       'construcao_ferramentas_jardim', 'fashion_roupa_masculina',
       'pet_shop', 'moveis_escritorio', 'market_place', 'eletronicos',
       'eletrodomesticos', 'artigos_de_festas', 'casa_conforto',
       'construcao_ferramentas_ferramentas', 'agro_industria_e_comercio',
       'moveis_colchao_e_estofado', 'livros_tecnicos', 'casa_construcao',
       'instrumentos_musicais', 'm

In [38]:
unknown_values = products[~products['product_category_name'].isin(category_name['product_category_name'])]
unknown_values

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,desconhecido,0,0,0,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,desconhecido,0,0,0,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,desconhecido,0,0,0,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,desconhecido,0,0,0,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,desconhecido,0,0,0,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,desconhecido,0,0,0,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,desconhecido,0,0,0,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,desconhecido,0,0,0,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,desconhecido,0,0,0,1300.0,45.0,16.0,45.0


In [39]:
new_row_1 = {'product_category_name':'desconhecido', 'product_category_name_english':'unknown'}
new_row_2 = {'product_category_name':'pc_gamer', 'product_category_name_english':'pc_gamer'}
new_row_3 = {'product_category_name':'portateis_cozinha_e_preparadores_de_alimentos', 'product_category_name_english':'portable_kitchen_food_preparers'}
category_name.loc[len(category_name)] = new_row_1
category_name.loc[len(category_name)] = new_row_2
category_name.loc[len(category_name)] = new_row_3

In [40]:
unknown_values = products[~products['product_category_name'].isin(category_name['product_category_name'])]
unknown_values

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [41]:
category_name.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

# Repasse para o SQL

In [ ]:
db_user = 'your_user'
db_password = 'your_password'
db_url = "your_url"

spark = SparkSession.builder \
  .appName("MySQL Integration") \
  .config("spark.jars", r"C:\\Users\\Gustavo\\Desktop\\Brazilian_ecommerce\\mysql-connector-j-8.0.33\\mysql-connector-j-8.0.33.jar") \
  .getOrCreate()

## Categoria

In [43]:
category_name_spark = spark.createDataFrame(category_name.copy())

category_name_spark.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [ ]:
category_name_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","category_name") \
    .mode("append") \
    .save()

In [45]:
print(f"A tabela 'category_name' foi migrada com sucesso ({len(category_name)} linhas adicionadas).")

A tabela 'category_name' foi migrada com sucesso (74 linhas adicionadas).


## Clientes

In [46]:
customers_spark = spark.createDataFrame(customers.copy())

customers_spark.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [ ]:
customers_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","customers") \
    .mode("append") \
    .save()

In [48]:
print(f"A tabela 'customers' foi migrada com sucesso ({len(customers)} linhas adicionadas).")

A tabela 'customers' foi migrada com sucesso (99441 linhas adicionadas).


## Vendedores

In [49]:
sellers_spark = spark.createDataFrame(sellers.copy())

sellers_spark.printSchema()


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: long (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [ ]:
sellers_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","sellers") \
    .mode("append") \
    .save()

In [51]:
print(f"A tabela 'sellers' foi migrada com sucesso ({len(sellers)} linhas adicionadas).")

A tabela 'sellers' foi migrada com sucesso (3095 linhas adicionadas).


## Pedidos

In [52]:
orders_spark = spark.createDataFrame(orders.copy())

orders_spark.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [ ]:
orders_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","orders") \
    .mode("append") \
    .save()

In [54]:
print(f"A tabela 'orders' foi migrada com sucesso ({len(orders)} linhas adicionadas).")

A tabela 'orders' foi migrada com sucesso (99441 linhas adicionadas).


## Produtos

In [55]:
products_spark = spark.createDataFrame(products.copy())

products_spark.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: long (nullable = true)
 |-- product_description_lenght: long (nullable = true)
 |-- product_photos_qty: long (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)



In [56]:
products_spark = products_spark.withColumn("product_id", col("product_id").cast("string")) \
                               .withColumn("product_category_name", col("product_category_name").cast("string")) \
                               .withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
                               .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
                               .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
                               .withColumn("product_weight_g", col("product_weight_g").cast("decimal(10,2)")) \
                               .withColumn("product_length_cm", col("product_length_cm").cast("decimal(10,2)")) \
                               .withColumn("product_height_cm", col("product_height_cm").cast("decimal(10,2)")) \
                               .withColumn("product_width_cm", col("product_width_cm").cast("decimal(10,2)"))

products_spark.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: decimal(10,2) (nullable = true)
 |-- product_length_cm: decimal(10,2) (nullable = true)
 |-- product_height_cm: decimal(10,2) (nullable = true)
 |-- product_width_cm: decimal(10,2) (nullable = true)



In [ ]:
products_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","products") \
    .mode("append") \
    .save()

In [58]:
print(f"A tabela 'products' foi migrada com sucesso ({len(products)} linhas adicionadas).")

A tabela 'products' foi migrada com sucesso (32951 linhas adicionadas).


## Itens do Pedido

In [59]:
order_items_spark = spark.createDataFrame(order_items.copy())

order_items_spark.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [60]:
order_items_spark = order_items_spark.withColumn("order_item_id", col("order_item_id").cast("int")) \
                                     .withColumn("price", col("price").cast("decimal(10,2)")) \
                                     .withColumn("freight_value", col("freight_value").cast("decimal(10,2)"))

order_items_spark.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)



In [ ]:
order_items_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","order_items") \
    .mode("append") \
    .save()

In [62]:
print(f"A tabela 'order_items' foi migrada com sucesso ({len(order_items)} linhas adicionadas).")

A tabela 'order_items' foi migrada com sucesso (112650 linhas adicionadas).


## Pagamentos do Pedido

In [63]:
order_payments_spark = spark.createDataFrame(order_payments.copy())

order_payments_spark.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: long (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: long (nullable = true)
 |-- payment_value: double (nullable = true)



In [ ]:
order_payments_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","order_payments") \
    .mode("append") \
    .save()

In [65]:
print(f"A tabela 'order_payments' foi migrada com sucesso ({len(order_payments)} linhas adicionadas).")

A tabela 'order_payments' foi migrada com sucesso (103886 linhas adicionadas).


## Review do Pedido

In [66]:
order_review_spark = spark.createDataFrame(order_review.copy())

order_review_spark.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: long (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [ ]:
order_review_spark.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("user",db_user) \
    .option("password", db_password) \
    .option("dbtable","order_review") \
    .mode("append") \
    .save()

In [68]:
print(f"A tabela 'order_review' foi migrada com sucesso ({len(order_review)} linhas adicionadas).")

A tabela 'order_review' foi migrada com sucesso (98410 linhas adicionadas).
